# Graph-Augmented Intelligence — Webinar
## Notebook 03: Pull Graph Features

Now that GDS algorithms have been executed in **Neo4j Aura** (via `02_aura_gds_guide`),
every Account node carries three new properties:

| Property | Algorithm | Fraud Signal |
|----------|-----------|-------------|
| `risk_score` | PageRank | Central accounts in money-flow networks |
| `community_id` | Louvain | Tightly connected clusters (fraud rings) |
| `similarity_score` | Node Similarity | Accounts sharing the same merchants |

This notebook:

```
Neo4j Aura ──► Spark Connector ──► Delta Lake ──► Feature Store ──► Training Table
```

**Next:** `04_train_model` trains baseline vs graph-augmented models and measures lift.

## 0. Configuration

In [ ]:
CATALOG   = f"graph_enriched_demo"
SCHEMA    = "graph_enriched_schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
}

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [ ]:
from pyspark.sql import functions as F

---
## 1. Read Graph Features from Neo4j

The Spark Connector reads the enriched Account nodes directly — all three
GDS-computed properties come along as columns.

In [ ]:
graph_features_df = (
    spark.read
    .format("org.neo4j.spark.DataSource")
    .options(**NEO4J_OPTS)
    .option("labels", "Account")
    .load()
    .select(
        F.col("account_id").cast("int"),
        F.col("risk_score").cast("double"),
        F.col("community_id").cast("long"),
        F.col("similarity_score").cast("double"),
    )
)

print(f"Read {graph_features_df.count()} Account nodes with graph features")
graph_features_df.show(10)

## 2. Write Graph Features to Delta

In [ ]:
FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.account_graph_features"

# Ensure account_id is NOT NULL so it can serve as a primary key
spark.sql(f"DROP TABLE IF EXISTS {FEATURES_TABLE}")

spark.sql(f"""
    CREATE TABLE {FEATURES_TABLE} (
        account_id     INT          NOT NULL,
        risk_score     DOUBLE,
        community_id   BIGINT,
        similarity_score DOUBLE,
        CONSTRAINT account_graph_features_pk PRIMARY KEY (account_id)
    )
""")

graph_features_df.writeTo(FEATURES_TABLE).append()

print(f"Written to {FEATURES_TABLE} (with PK constraint)")

## 3. Register in Feature Store (Unity Catalog)

Makes graph features discoverable and joinable with any training dataset
by `account_id`.

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Read back from Delta (not the Neo4j source) to avoid Spark V2 push-down issues
graph_features_delta = spark.table(FEATURES_TABLE)

fe.write_table(
    name=FEATURES_TABLE,
    df=graph_features_delta,
    mode="merge",
)

print(f"Feature Store updated: {graph_features_delta.count()} rows")

---
## 4. Build the Full Training Table

Join original tabular features with graph-derived features to create
the complete training dataset.

In [ ]:
# Join accounts with account_labels to bring in is_fraud as the training label.
# The operational accounts table has no is_fraud column; labels live in account_labels.
accounts_df = (
    spark.table(f"{CATALOG}.{SCHEMA}.accounts")
    .join(spark.table(f"{CATALOG}.{SCHEMA}.account_labels"), "account_id", "left")
)
graph_feat  = spark.table(FEATURES_TABLE)

# Tabular features from transactions
txn_features = (
    spark.table(f"{CATALOG}.{SCHEMA}.transactions")
    .groupBy("account_id")
    .agg(
        F.count("*").alias("txn_count"),
        F.round(F.avg("amount"), 2).alias("avg_txn_amount"),
        F.round(F.stddev("amount"), 2).alias("std_txn_amount"),
        F.round(F.max("amount"), 2).alias("max_txn_amount"),
        F.countDistinct("merchant_id").alias("unique_merchants"),
        F.round(F.avg("txn_hour").cast("double"), 2).alias("avg_txn_hour"),
        F.round(F.sum(F.when(F.col("txn_hour").between(0, 5), 1).otherwise(0)) / F.count("*"), 4)
            .alias("night_txn_ratio"),
    )
)

# P2P transfer features
p2p_features = (
    spark.table(f"{CATALOG}.{SCHEMA}.account_links")
    .groupBy(F.col("src_account_id").alias("account_id"))
    .agg(
        F.count("*").alias("p2p_out_count"),
        F.round(F.avg("amount"), 2).alias("avg_p2p_amount"),
    )
)

# Full training table
training_df = (
    accounts_df
    .join(txn_features,  "account_id", "left")
    .join(p2p_features,  "account_id", "left")
    .join(graph_feat,    "account_id", "left")
    .fillna(0)
)

TRAINING_TABLE = f"{CATALOG}.{SCHEMA}.training_dataset"
(training_df
 .write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(TRAINING_TABLE)
)

print(f"Training table: {training_df.count()} rows, {len(training_df.columns)} columns")
training_df.printSchema()

## 5. Preview — Feature Correlations with Fraud

In [ ]:
display(
    training_df
    .groupBy("is_fraud")
    .agg(
        F.round(F.avg("risk_score"), 6).alias("avg_risk_score"),
        F.round(F.avg("similarity_score"), 4).alias("avg_similarity"),
        F.round(F.avg("txn_count"), 1).alias("avg_txn_count"),
        F.round(F.avg("unique_merchants"), 1).alias("avg_unique_merchants"),
        F.round(F.avg("night_txn_ratio"), 4).alias("avg_night_ratio"),
        F.round(F.avg("p2p_out_count"), 1).alias("avg_p2p_out"),
    )
)

---

### Flow so far

| Step | Notebook | Where |
|------|----------|-------|
| 1. Generate synthetic fraud data | `00_setup_and_data` | Databricks |
| 2. Push to Neo4j Aura | `01_neo4j_ingest` | Databricks |
| 3. Run GDS algorithms | `02_aura_gds_guide` | Neo4j Aura Workspace |
| 4. Pull features & build training table | **`03_pull_gold_tables`** | Databricks |
| 5. Train & evaluate models | `04_train_model` | Databricks |